In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

from src.cleaning import clean_transactions

from src.merchant_extraction import (
    load_merchant_map,
    cleanup_description,
    normalize_merchant
)


from src.eda import (
    spending_by_category,
    average_monthly_spending_by_category,
    spending_by_merchant,
    average_monthly_spending_by_merchant,
    monthly_spending,
    average_monthly_spending
)

from src.visualization import (
    plot_avg_monthly_spend_by_merchant,
    plot_monthly_spending,
    plot_average_spend_by_month,
    plot_top_merchants_last_12_months,
    plot_top_merchants_by_category
)

In [2]:
df = clean_transactions(
    "../data/Discover_Transaction_History.csv"
)

print(f"Transactions: {len(df):,}")

df.head()

Transactions: 3,286


/Users/mafphd/personalprojects/src/cleaning.py:53: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["trans_date"] = pd.to_datetime(
/Users/mafphd/personalprojects/src/cleaning.py:59: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["post_date"] = pd.to_datetime(


,trans_date,post_date,description,amount,category,desc_clean,month,day_of_week,is_weekend
0,2023-10-17,2023-10-19,WALGREENS #10196 WAUWATOSA WI,13.70,Merchandise,WALGREENS WAUWATOSA WI,2023-10,Tuesday,False
1,2023-10-18,2023-10-19,5-365 FOOD SERVICE TROY MI,3.05,Restaurants,FOOD SERVICE TROY MI,2023-10,Wednesday,False
2,2023-10-18,2023-10-19,QUALITY HEATING AND SHEE BROOKFIELD WI,79.65,Services,QUALITY HEATING AND SHEE BROOKFIELD WI,2023-10,Wednesday,False
3,2023-10-18,2023-10-19,SOMMER'S INC. MEQUON WI649149,114.59,Automotive,SOMMER S INC MEQUON WI649149,2023-10,Wednesday,False
4,2023-10-19,2023-10-19,MILTOWN EATS LLC 4144349799 WI,30.00,Supermarkets,MILTOWN EATS LLC WI,2023-10,Thursday,False


In [9]:
unique_descriptions = (
    df["description"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

unique_descriptions_df = pd.DataFrame({
    "description": unique_descriptions
})

print(
    f"Unique descriptions: "
    f"{len(unique_descriptions_df):,}"
)

unique_descriptions_df.head()

Unique descriptions: 1,403


,description
0,#492028TOPPIZTOSA 4142574002 WI
1,#492028TOPPIZTOSA WAUWATOSA WI
2,#5993 VILLA IK MAYFAIR WAUWATOSA WI
3,077 HARDEES SHEBOYGAN SHEBOYGAN WI
4,2LEVY AT FISERV FORUM MILWAUKEE WI


In [10]:
unique_descriptions_df.to_csv(
    "../outputs/unique_descriptions.csv",
    index=False
)

#Use external merchant_lookup.csv file based on unique_descriptions_df

In [6]:
merchant_lookup = pd.read_csv(
    "../data/merchant_lookup.csv"
)

merchant_lookup.head()

,description,merchant_clean,Unnamed: 2,Unnamed: 3
0,#492028TOPPIZTOSA 4142574002 WI,TOPPIZTOSA,NaN,NaN
1,#492028TOPPIZTOSA WAUWATOSA WI,TOPPIZTOSA,NaN,NaN
2,#5993 VILLA IK MAYFAIR WAUWATOSA WI,VILLA IK MAYFAIR,NaN,NaN
3,077 HARDEES SHEBOYGAN SHEBOYGAN WI,HARDEE'S,NaN,NaN
4,2LEVY AT FISERV FORUM MILWAUKEE WI,LEVY AT FISERV FORUM,NaN,NaN


In [25]:
merchant_lookup = merchant_lookup[["description", "merchant_clean"]]

analysis_df = df.merge(
    merchant_lookup,
    on="description",
    how="left"
)

analysis_df.head()

,trans_date,post_date,description,amount,category,desc_clean,month,day_of_week,is_weekend,merchant_clean
0,2023-10-17,2023-10-19,WALGREENS #10196 WAUWATOSA WI,13.70,Merchandise,WALGREENS WAUWATOSA WI,2023-10,Tuesday,False,WALGREENS
1,2023-10-18,2023-10-19,5-365 FOOD SERVICE TROY MI,3.05,Restaurants,FOOD SERVICE TROY MI,2023-10,Wednesday,False,365 FOOD SERVICE
2,2023-10-18,2023-10-19,QUALITY HEATING AND SHEE BROOKFIELD WI,79.65,Services,QUALITY HEATING AND SHEE BROOKFIELD WI,2023-10,Wednesday,False,QUALITY HEATING & SHEET METAL
3,2023-10-18,2023-10-19,SOMMER'S INC. MEQUON WI649149,114.59,Automotive,SOMMER S INC MEQUON WI649149,2023-10,Wednesday,False,SOMMER'S
4,2023-10-19,2023-10-19,MILTOWN EATS LLC 4144349799 WI,30.00,Supermarkets,MILTOWN EATS LLC WI,2023-10,Thursday,False,MILTOWN EATS


In [26]:
analysis_df = analysis_df[analysis_df['merchant_clean'] != 'APPLIANCE GALLERY']
analysis_df.head()

,trans_date,post_date,description,amount,category,desc_clean,month,day_of_week,is_weekend,merchant_clean
0,2023-10-17,2023-10-19,WALGREENS #10196 WAUWATOSA WI,13.70,Merchandise,WALGREENS WAUWATOSA WI,2023-10,Tuesday,False,WALGREENS
1,2023-10-18,2023-10-19,5-365 FOOD SERVICE TROY MI,3.05,Restaurants,FOOD SERVICE TROY MI,2023-10,Wednesday,False,365 FOOD SERVICE
2,2023-10-18,2023-10-19,QUALITY HEATING AND SHEE BROOKFIELD WI,79.65,Services,QUALITY HEATING AND SHEE BROOKFIELD WI,2023-10,Wednesday,False,QUALITY HEATING & SHEET METAL
3,2023-10-18,2023-10-19,SOMMER'S INC. MEQUON WI649149,114.59,Automotive,SOMMER S INC MEQUON WI649149,2023-10,Wednesday,False,SOMMER'S
4,2023-10-19,2023-10-19,MILTOWN EATS LLC 4144349799 WI,30.00,Supermarkets,MILTOWN EATS LLC WI,2023-10,Thursday,False,MILTOWN EATS


In [27]:
pd.set_option("display.max_rows", 100)
merchant_review = (
    analysis_df
    .groupby("merchant_clean")
    .agg(
        transaction_count=("amount", "count"),
        total_spend=("amount", "sum"),
        sample_description=("description", "first")
    )
    .reset_index()
    .sort_values(
        "total_spend",
        ascending=False
    )
)

display(
    merchant_review.head(100)
)

,merchant_clean,transaction_count,total_spend,sample_description
86,CENTRAL BARK,189,8127.07,CENTRAL BARK-BROOKFIELD 262-781-5554 WI
303,LEGACY GYM MKE,75,7065.40,LEGACY GYM MKE 262-989-6696 WI
10,AAA INSURANCE,7,5095.03,AAA INSURANCE GW EFT 800-222-6424 MI
479,SENDIK'S,131,4717.82,SENDIK'S WAUWATOSA WAUWATOSA WIAPPLE PAY ENDIN...
355,MILTOWN EATS,90,4436.98,MILTOWN EATS LLC 4144349799 WI
276,JULIE'S PARK CAFE & MOTEL,6,3955.48,JULIES PARK CAFE & MOTEL FISH CREEK WI
342,METCALFE'S MARKET,47,3241.84,METCALFE MARKET TOSA WAUWATOSA WI
25,AMTRAK,8,3204.25,AMTRAK MOBILE 202-906-3000 DCAPPLE PAY ENDING ...
503,SOMMER'S AUTOMOTIVE,2,3138.67,SOMMER'S AUTOMOTIVE - 262-242-0100 WI173412795...
359,MILWAUKEE ELECTRIC TOOL,10,2819.33,SQ *MILWAUKEE ELECTRIC BROOKFIELD WI0002305843...


In [ ]:
# --------------------------------------------------
# Merchant Summary
# --------------------------------------------------

total_months = (
    analysis_df["month"]
    .nunique()
)

merchant_summary = (
    analysis_df
    .groupby("merchant_clean")
    .agg(
        transaction_count=("amount", "count"),
        active_months=("month", "nunique"),
        total_spend=("amount", "sum"),
        average_transaction=("amount", "mean")
    )
    .reset_index()
)

# Average spend across ALL months in analysis period
merchant_summary["avg_monthly_spend"] = (
    merchant_summary["total_spend"]
    / total_months
)

# Fraction of months merchant appears
merchant_summary["monthly_frequency"] = (
    merchant_summary["active_months"]
    / total_months
)

merchant_summary = (
    merchant_summary
    .sort_values(
        "avg_monthly_spend",
        ascending=False
    )
)

display(
    merchant_summary.head(50)
)

In [ ]:
plot_avg_monthly_spend_by_merchant(
    merchant_summary,
    top_n=20
)

In [ ]:
monthly_totals = (
    analysis_df
    .groupby("month")["amount"]
    .sum()
    .reset_index(
        name="monthly_spend"
    )
)

In [ ]:
plot_monthly_spending(
    monthly_totals
)

In [ ]:
plot_average_spend_by_month(
    analysis_df
)

In [ ]:
plot_top_merchants_last_12_months(
    analysis_df,
    top_n=20
)

In [ ]:
plot_top_merchants_by_category(
    analysis_df,
    top_n=20
)

In [ ]:
from src.visualization import (
    plot_merchant_frequency_vs_spend
)

plot_merchant_frequency_vs_spend(
    merchant_summary,
    min_monthly_spend=20,
    top_n_labels=30
)